In [1]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master("spark://spark-master:7077")
         .appName("SahlPay-Partie1")
         .getOrCreate())

sc = spark.sparkContext
sc.setLogLevel("WARN")

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ab039c7f-9c53-4e37-b138-7e702590a3a6;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.8 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.8 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.5 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#ha

In [2]:
print("Version de Spark   :", spark.version)
print("Master             :", sc.master)
print("appName            :", sc.appName)
print("Coeurs disponibles :", sc.defaultParallelism)

nb_executors = sc._jsc.sc().getExecutorMemoryStatus().size() - 1   # -1 = sans le driver
print("Executors actifs   :", nb_executors)

Version de Spark   : 3.5.8
Master             : spark://spark-master:7077
appName            : SahlPay-Partie1
Coeurs disponibles : 4
Executors actifs   : 2


In [4]:
test_df = spark.createDataFrame(
    [("Rabat", 320.0), ("Casablanca", 150.5), ("Fes", 80.0)],
    ["city", "amount_mad"])

chemin = "s3a://lake/tests/partie1/"
test_df.write.mode("overwrite").parquet(chemin)
spark.read.parquet(chemin).show()

+----------+----------+
|      city|amount_mad|
+----------+----------+
|Casablanca|     150.5|
|     Rabat|     320.0|
|       Fes|      80.0|
+----------+----------+



In [5]:
msg = spark.createDataFrame([("cle-1", '{"test": "bonjour Kafka"}')], ["key", "value"])

(msg.write.format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("topic", "test-partie1")
    .save())

lu = (spark.read.format("kafka")
      .option("kafka.bootstrap.servers", "kafka:9092")
      .option("subscribe", "test-partie1")
      .option("startingOffsets", "earliest")
      .load())

lu.selectExpr("CAST(key AS STRING)", "CAST(value AS STRING)", "partition", "offset").show(truncate=False)

26/11/11 01:43:40 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
[Stage 10:=============================>                            (1 + 1) / 2]

+-----+-------------------------+---------+------+
|key  |value                    |partition|offset|
+-----+-------------------------+---------+------+
|cle-1|{"test": "bonjour Kafka"}|1        |0     |
+-----+-------------------------+---------+------+



In [6]:
spark.stop()